# S&P 500 data for the group ML exercise

Everyone uses the same **nine features**, **next-day return target**, and **date splits**. Each member trains a regression model, then ranks its predictions within each date.

Run the cells in order from a clone of this repository. In Colab, first run `!git clone https://github.com/Simon666Z/financial-engineering-club-fall26.git`, then `%cd financial-engineering-club-fall26`.

The stock list is the S&P 500 list downloaded for this snapshot, not historical membership. This is a learning dataset. The close-to-close target measures prediction quality; a trading backtest must use prices available after the signal is formed.


In [ ]:
%pip install -q "pandas>=2.1,<3" numpy requests yfinance lxml


## 1. Settings

Keep the settings the same for everyone. Cached raw inputs are reused, so rerunning the notebook does not silently change the stock list or downloaded prices. Set `REFRESH_DATA = True` only when the group intentionally creates a new snapshot.


In [ ]:
from pathlib import Path
from io import StringIO
import hashlib
import sys

import pandas as pd
import requests
import yfinance as yf

# Find the repository whether Jupyter started in its root or notebooks folder.
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/data/preparation.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Clone the full repository first; see the first cell.")
sys.path.insert(0, str(ROOT))
from src.data.preparation import (
    FEATURES, prepare_price_panel, make_model_dataset, add_date_splits,
)

START = "2016-01-01"
END = "2026-01-01"  # exclusive: includes all of 2025
REFRESH_DATA = False
TRAIN_END = "2022-12-31"
VALIDATION_END = "2023-12-31"
features = FEATURES.copy()

RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CONSTITUENTS_FILE = RAW_DIR / f"sp500_{START}_{END}.csv"
PRICES_FILE = RAW_DIR / f"prices_{START}_{END}.csv"
USE_CACHE = (
    not REFRESH_DATA and CONSTITUENTS_FILE.exists() and PRICES_FILE.exists()
)


## 2. Load the stock list and daily prices

The first run needs internet access and downloads 2016–2025 prices. Later runs use the cached inputs. Failed downloads remain visible in the cleaning summary rather than receiving fabricated prices.


In [ ]:
if USE_CACHE:
    sp500 = pd.read_csv(CONSTITUENTS_FILE)
else:
    response = requests.get(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        headers={"User-Agent": "Mozilla/5.0"}, timeout=30,
    )
    response.raise_for_status()
    required = ["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]
    tables = pd.read_html(StringIO(response.text))
    sp500 = next((table for table in tables if set(required).issubset(table.columns)), None)
    if sp500 is None:
        raise ValueError("The S&P 500 table did not contain the expected columns.")
    sp500 = sp500[required].copy()
    sp500["Ticker"] = sp500["Symbol"].str.strip().str.replace(".", "-", regex=False)

metadata_columns = ["Ticker", "Security", "GICS Sector", "GICS Sub-Industry"]
sp500 = sp500.sort_values("Ticker").reset_index(drop=True)
tickers = sp500["Ticker"].tolist()
print("Securities in the shared stock list:", len(tickers))
display(sp500.head())


In [ ]:
if USE_CACHE:
    raw_prices = pd.read_csv(
        PRICES_FILE, header=[0, 1], index_col=0, parse_dates=[0], float_precision="round_trip"
    )
    print("Loaded cached prices:", PRICES_FILE.name)
else:
    raw_prices = yf.download(
        tickers=tickers, start=START, end=END, interval="1d",
        auto_adjust=False, group_by="ticker", multi_level_index=True,
        keepna=True, ignore_tz=True, threads=True, progress=True,
    )
    if raw_prices is None or raw_prices.empty:
        raise ValueError("No prices were downloaded. Rerun this cell after checking the connection.")

if not isinstance(raw_prices.columns, pd.MultiIndex) or raw_prices.columns.nlevels != 2:
    raise ValueError("Expected ticker/field columns from Yahoo Finance.")
if "Adj Close" not in raw_prices.columns.get_level_values(1):
    raise ValueError("Adjusted Close is missing; keep auto_adjust=False.")

quote_columns = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
expected_columns = pd.MultiIndex.from_product(
    [tickers, quote_columns], names=["Ticker", "Field"]
)
raw_prices = raw_prices.reindex(columns=expected_columns).sort_index()
raw_prices.index = pd.to_datetime(raw_prices.index)
raw_prices.index.name = "Date"
if raw_prices.empty or raw_prices.index.has_duplicates:
    raise ValueError("Price dates must be nonempty and unique.")
if (raw_prices.index < pd.Timestamp(START)).any() or (raw_prices.index >= pd.Timestamp(END)).any():
    raise ValueError("Cached price dates do not match START and END.")

# Write the pair only after a successful download and structural check.
if not USE_CACHE:
    raw_prices.to_csv(PRICES_FILE)
    sp500.to_csv(CONSTITUENTS_FILE, index=False)

raw_panel = (
    raw_prices.stack(level="Ticker", future_stack=True)
    .rename_axis(["Date", "Ticker"])
    .reset_index()
)
raw_panel.columns.name = None
print("Raw price rows:", len(raw_panel))


## 3. Clean the quotes and build the original features

The shared preparation function checks duplicate records, missing or infinite values, positive prices and volume, and sensible High/Low bounds. Invalid quotes are masked, not forward-filled. Keeping their dates prevents a missing day from turning a next-day target into a multi-day return.

The original features are unchanged: one-day return; 5-, 20-, and 60-day momentum; 5- and 20-day volatility; relative volume; daily price range; and intraday return. Rolling features need a complete window. The formulas and checks are in `src/data/preparation.py`.


In [ ]:
panel, cleaning_summary = prepare_price_panel(raw_panel, sp500[metadata_columns])
display(pd.Series(
    {key: value for key, value in cleaning_summary.items()
     if key not in {"invalid_reason_counts", "reason_counts_overlap"}},
    name="row count",
))
display(pd.Series(cleaning_summary["invalid_reason_counts"], name="flagged rows (counts overlap)"))

valid_per_stock = panel.groupby("Ticker")["QuoteValid"].sum()
unavailable_tickers = valid_per_stock.index[valid_per_stock.eq(0)].tolist()
print("Tickers with no usable quotes:", unavailable_tickers)


## 4. One shared model table

Only rows with valid current prices and all nine finite features are kept. Eligibility uses information available on that date.

`target` stays the next trading session's adjusted close-to-close return, expressed as a decimal. `target_rank` is its percentile rank within the date, with higher values meaning higher realized returns. It is provided for checking rankings, not as an input feature.

Rows whose future return is unavailable stay in the table with a missing target. They can still receive predictions; exclude them only when fitting or scoring a model.


In [ ]:
model_data = make_model_dataset(panel, labeled_only=False)
model_data = add_date_splits(model_data, TRAIN_END, VALIDATION_END)
columns = ["Date", "Ticker", "Split", "LabelEndDate", *features, "target", "target_rank"]
model_data = model_data[columns].copy()

if model_data.empty:
    raise ValueError("No rows have complete features. Check the cleaning summary.")
if not {"train", "validation", "test"}.issubset(set(model_data["Split"])):
    raise ValueError("One of the shared periods is empty. Check dates and downloaded history.")
assert not model_data.duplicated(["Date", "Ticker"]).any()
assert model_data[features].notna().all().all()
display(model_data.groupby("Split").agg(rows=("Ticker", "size"), known_targets=("target", "count")))
display(model_data.head())


## 5. Prepare the learning inputs

Train through 2022, choose model settings using 2023, and compare final results on 2024–2025. All stocks on a date stay together. The last signal date at a split boundary is marked `purged` if its label belongs to the following period.

Use **only `features`** for model inputs. Fit scaling or other learned preprocessing on training data only. Do not use `target`, `target_rank`, `LabelEndDate`, or `Split` as features.


In [ ]:
train = model_data.loc[model_data["Split"].eq("train")].dropna(subset=["target"])
validation = model_data.loc[model_data["Split"].eq("validation")].dropna(subset=["target"])
test_signals = model_data.loc[model_data["Split"].eq("test")].copy()
test = test_signals.dropna(subset=["target"])

X_train, y_train = train[features], train["target"]
X_validation, y_validation = validation[features], validation["target"]
X_test, y_test = test[features], test["target"]
print("Labeled train / validation / test rows:", len(train), len(validation), len(test))
print("All eligible test prediction rows:", len(test_signals))


## 6. Save and share the same dataset

The group leader runs this notebook once and shares the resulting CSV. Everyone should use that file and the same printed checksum. Generated data files are excluded from Git.

For the shared backtest proposal, see `docs/backtesting.md`. Keep the cached quote inputs alongside this export: the portfolio simulator will need the next session's actual Open and Close prices.


In [ ]:
OUTPUT_FILE = PROCESSED_DIR / "model_data.csv"
model_data.to_csv(OUTPUT_FILE, index=False)
digest = hashlib.sha256()
with OUTPUT_FILE.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)
checksum = digest.hexdigest()
print("Saved:", OUTPUT_FILE)
print("SHA256:", checksum)

# Later, each member can load the shared table with this one line:
# model_data = pd.read_csv(OUTPUT_FILE, parse_dates=["Date", "LabelEndDate"])
